# Détection d’objets avec YOLO11n

Localiser les objets et prédire leur classe sur une photographie réelle. Une seule image, résolution 640, modèle nano. La détection ne maintient pas des identités temporelles comme le tracking.

**Statut : code préparé et vérifié syntaxiquement, inférence non exécutée dans cet environnement.** Exécuter de haut en bas dans Colab ou Jupyter. Les sorties sont volontairement vides, aucun résultat inventé. Petits tests sans entraînement. Python 3.10–3.12 recommandé.

## 1. Bibliothèques
Installer seulement les dépendances du sujet. Le premier lancement télécharge les poids COCO.

In [ ]:
%pip install "ultralytics>=8.3,<9" "matplotlib>=3.8,<4" "pillow>=10,<13"

In [ ]:
from pathlib import Path
import time, urllib.request
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
DATA=Path('data'); DATA.mkdir(exist_ok=True)
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)
def download(url, name):
    path=DATA/name
    if not path.exists():
        with urllib.request.urlopen(url, timeout=60) as r:
            path.write_bytes(r.read())
    return path


## 2. Image et modèle
Photo de démonstration publique Ultralytics (bus). Une image personnelle peut remplacer le fichier, sans changer la suite.

In [ ]:
from ultralytics import YOLO
import torch
path=download('https://ultralytics.com/images/bus.jpg','bus.jpg')
image=Image.open(path).convert('RGB')
model=YOLO('yolo11n.pt')
device=0 if torch.cuda.is_available() else 'cpu'
print('Appareil :',device,'| image :',image.size)
plt.imshow(image); plt.axis('off'); plt.show()


## 3. Inférence et visualisation
Le score de confiance n’est pas une précision globale. Les temps dépendent du matériel, de la taille et du chargement.

In [ ]:
if torch.cuda.is_available(): torch.cuda.synchronize()
t0=time.perf_counter()
result=model.predict(image,imgsz=640,conf=0.25,device=device,verbose=False)[0]
if torch.cuda.is_available(): torch.cuda.synchronize()
print(f'Inférence et préparation : {time.perf_counter()-t0:.3f} s')
print('Boîtes :',len(result.boxes))
for box in result.boxes:
    print(model.names[int(box.cls.item())],round(float(box.conf.item()),3),box.xyxy.tolist()[0])
plt.figure(figsize=(10,8));plt.imshow(result.plot()[...,::-1]);plt.axis('off');plt.show()
result.save(filename=str(OUT/'detections.jpg'))


## 4. Effet du seuil et évaluation
Sans annotations de référence, nous ne calculons pas de mAP. Comparer simplement le nombre de boîtes aux seuils 0,25 et 0,60. Pour une vraie évaluation utiliser un dataset annoté et `model.val(data="dataset.yaml")`.

In [ ]:
strict=model.predict(image,imgsz=640,conf=0.60,device=device,verbose=False)[0]
print({'seuil_025':len(result.boxes),'seuil_060':len(strict.boxes)})
assert result.orig_shape==strict.orig_shape
assert len(strict.boxes)<=len(result.boxes)


## Questions de compréhension
1. Détection vs tracking ?
2. Que représente une boîte xyxy ?
3. Effet du seuil de confiance ?
4. Pourquoi aucune mAP ici ?

## Bibliographie
- https://docs.ultralytics.com/models/yolo11/
- https://docs.ultralytics.com/modes/predict/
- Image : https://ultralytics.com/images/bus.jpg
Les dépendances et poids ont leurs propres licences, notamment Ultralytics AGPL-3.0 ou licence commerciale selon usage.